# Database - Creation
    - Author: Santiago Avella. Git hub: https://github.com/TiagoMimi

In [15]:
import sqlite3
import pandas as pd
from pathlib import Path
from graphviz import Digraph
import json

## Load the data

In [16]:
# Current working directory
script_path = Path.cwd()

# Find the project folder
project_path = script_path

while project_path.name != "github":
    project_path = project_path.parent


In [22]:
data_path = project_path / "saber11_db" / "data" / "joined" 

# Read teacher data types
with open(
    data_path / "data_types_teacher.json",
    "r",
    encoding="utf-8"
) as file:
    data_type_teacher = json.load(file)


# Read student data types
with open(
    data_path / "data_types_student.json",
    "r",
    encoding="utf-8"
) as file:
    data_type_student = json.load(file)


# List of CSV files
file_list = [
    data_path / "teacher_schedule.csv",
    data_path / "teacher_rank.csv",
    data_path / "teacher_employment.csv",
    data_path / "teacher_education.csv",
    data_path / "teacher_clei.csv",
    data_path / "teacher_age.csv",
    data_path / "teacher_academic_assignment.csv",
    data_path / "student_info.csv",
    data_path / "socioeconomic_info.csv",
    data_path / "school_info.csv",
    data_path / "result_info.csv"
]


# DataFrame names
df_names = [
    "df_teacher_schedule",
    "df_teacher_rank",
    "df_teacher_employment",
    "df_teacher_education",
    "df_teacher_clei",
    "df_teacher_age",
    "df_teacher_academic_assignment",
    "df_student_info",
    "df_socioeconomic_info",
    "df_school_info",
    "df_result_info",
]


# Read CSV files using the data types stored in the JSON files
dataframes = {}

for name, file in zip(df_names, file_list):

    # Get table name
    table_name = name.replace("df_", "")

    # Select the corresponding data type dictionary
    if table_name.startswith("teacher_"):
        dtype_mapping = data_type_teacher[table_name]
    else:
        dtype_mapping = data_type_student[table_name]

    # Read CSV
    dataframes[name] = pd.read_csv(
        file,
        sep=";",
        dtype=dtype_mapping,
        low_memory=False
    )


# Assign DataFrames
df_teacher_schedule = dataframes["df_teacher_schedule"]
df_teacher_rank = dataframes["df_teacher_rank"]
df_teacher_employment = dataframes["df_teacher_employment"]
df_teacher_education = dataframes["df_teacher_education"]
df_teacher_clei = dataframes["df_teacher_clei"]
df_teacher_age = dataframes["df_teacher_age"]
df_teacher_academic_assignment = dataframes["df_teacher_academic_assignment"]

df_student_info = dataframes["df_student_info"]
df_socioeconomic_info = dataframes["df_socioeconomic_info"]
df_school_info = dataframes["df_school_info"]
df_result_info = dataframes["df_result_info"]

## Load the data to sql

In [23]:
import duckdb

# Path where the database will be stored
database_path = project_path / "saber11_db" / "data" / "database" / "saber11_teacher.db"

# Create parent directories if they don't exist
database_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Create or connect to the DuckDB database
conn = duckdb.connect(database_path)


# DataFrames to load into the database
dataframes = {
    "teacher_schedule": df_teacher_schedule,
    "teacher_rank": df_teacher_rank,
    "teacher_employment": df_teacher_employment,
    "teacher_education": df_teacher_education,
    "teacher_clei": df_teacher_clei,
    "teacher_age": df_teacher_age,
    "teacher_academic_assignment": df_teacher_academic_assignment,
    "student_info": df_student_info,
    "socioeconomic_info": df_socioeconomic_info,
    "school_info": df_school_info,
    "result_info": df_result_info
}


# Create a table for each DataFrame
for table_name, df in dataframes.items():

    # Register DataFrame temporarily in DuckDB
    conn.register("temp_df", df)

    # Create or replace permanent table
    conn.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM temp_df
    """)


print("Database created successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Database created successfully.


## Test

In [34]:
query = """
SELECT *
FROM student_info
LIMIT 10
"""

df = conn.execute(query).df()
df

,student_id,estu_agregado,school_institution_id,school_campus_id,estu_dedicacioninternet,estu_dedicacionlecturadiaria,estu_depto_reside,estu_discapacidad,estu_estudiante,estu_etnia,...,estu_pilopaga,estu_privado_libertad,estu_repite,estu_tieneetnia,estu_tipodocumento,estu_tiporemuneracion,estu_cod_reside_depto,estu_cod_reside_mcpio,estu_mcpio_reside,estu_pais_reside
0,SB11201510000485,S,311769000921,311769000921,None,None,BOGOTA,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,CC,None,11,11001,BOGOTÁ D.C.,COLOMBIA
1,SB11201510115564,S,376250000871,376250000871,None,None,VALLE,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,TI,None,76,76250,EL DOVIO,COLOMBIA
2,SB11201510097089,S,311001105928,311001105928,None,None,BOGOTA,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,TI,None,11,11001,BOGOTÁ D.C.,COLOMBIA
3,SB11201510115560,S,376250000871,376250000871,None,None,VALLE,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,CC,None,76,76622,ROLDANILLO,COLOMBIA
4,SB11201510002784,S,308001004209,308001004209,None,None,ATLANTICO,None,ESTUDIANTE,Sikuani,...,NO,N,NaN,None,TI,None,8,8001,BARRANQUILLA,COLOMBIA
5,SB11201510097623,S,305001024544,305001024544,None,None,ANTIOQUIA,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,CC,None,5,5001,MEDELLIN,COLOMBIA
6,SB11201510065897,S,376001031031,376001031031,None,None,VALLE,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,CC,None,76,76001,CALI,COLOMBIA
7,SB11201510115562,S,376250000871,376250000871,None,None,VALLE,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,CC,None,76,76250,EL DOVIO,COLOMBIA
8,SB11201510098628,S,311001104778,311001104778,None,None,BOGOTA,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,CC,None,11,11001,BOGOTÁ D.C.,COLOMBIA
9,SB11201510032092,S,311001092320,311001092320,None,None,BOGOTA,None,ESTUDIANTE,Ninguno,...,NO,N,NaN,None,TI,None,11,11001,BOGOTÁ D.C.,COLOMBIA


In [35]:
df["estu_dedicacioninternet"].value_counts()

Series([], Name: count, dtype: int64)

In [29]:
query = """
SELECT *
FROM school_info
LIMIT 10
"""

df = conn.execute(query).df()

df

,school_campus_id,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,school_institution_id,cole_cod_depto_ubicacion,cole_cod_mcpio_ubicacion,cole_codigo_icfes,cole_depto_ubicacion,cole_genero,cole_jornada,cole_mcpio_ubicacion,cole_naturaleza,cole_nombre_establecimiento,cole_nombre_sede,cole_sede_principal
0,1,URBANO,None,OTRO,ACADÉMICO,1,11,11001,722157,BOGOTÁ,MIXTO,MAÑANA,BOGOTÁ D.C.,OFICIAL,NORMAL SUPERIOR MARIA AUXILIADORA,INSTITUTO COLOMBIANO PARA LA EVALUACIÓN DE LA ...,S
1,105001000001,URBANO,N,A,ACADÉMICO,105001000001,5,5001,113381,ANTIOQUIA,MIXTO,TARDE,MEDELLÍN,OFICIAL,INSTITUCION EDUCATIVA FE Y ALEGRIA JOSE MARIA ...,INST EDUC FE Y ALEGRIA JOSE MARIA VELAZ,S
2,105001000043,URBANO,N,A,ACADÉMICO,105001000043,5,5001,159251,ANTIOQUIA,MIXTO,NOCHE,MEDELLÍN,OFICIAL,INSTITUCION EDUCATIVA BARRIO SANTA CRUZ,INST EDUC BARRIO SANTA CRUZ,S
3,105001000108,URBANO,N,A,TÉCNICO/ACADÉMICO,105001000108,5,5001,724211,ANTIOQUIA,FEMENINO,UNICA,MEDELLÍN,OFICIAL,INSTITUCION EDUCATIVA CENTRO FORMATIVO DE ANTI...,INST EDUC CEFA,S
4,105001000132,URBANO,None,A,ACADÉMICO,105001000132,5,5001,133272,ANTIOQUIA,MIXTO,MAÑANA,MEDELLÍN,OFICIAL,INST EDUC JOSE MARIA BERNAL,INST EDUC JOSE MARIA BERNAL,S
5,105001000141,URBANO,N,A,ACADÉMICO,105001000141,5,5001,113498,ANTIOQUIA,MIXTO,MAÑANA,MEDELLÍN,OFICIAL,INSTITUCION EDUCATIVA PRESBITERO CAMILO TORRES...,INST EDUC CAMILO TORRES RESTREPO,S
6,105001000167,URBANO,None,A,TÉCNICO/ACADÉMICO,105001000167,5,5001,127027,ANTIOQUIA,MIXTO,MAÑANA,MEDELLIN,NO OFICIAL,COL LA PASTORA-EN ADMINISTRACION (AC),COL LA PASTORA-EN ADMINISTRACION (AC),S
7,105001000175,URBANO,N,A,TÉCNICO/ACADÉMICO,105001000175,5,5001,727354,ANTIOQUIA,MIXTO,UNICA,MEDELLÍN,OFICIAL,INST EDUC GABRIEL RESTREPO MORENO,INST EDUC GABRIEL RESTREPO MORENO,S
8,105001000191,URBANO,N,A,ACADÉMICO,305001015686,5,5001,199950,ANTIOQUIA,MIXTO,MAÑANA,MEDELLÍN,OFICIAL,INSTITUCION EDUCATIVA MAESTRO ARENAS BETANCUR,SEC ESC IMPERIO DEL JAPON,N
9,105001000205,URBANO,None,A,ACADÉMICO,105001000205,5,5001,138131,ANTIOQUIA,MIXTO,MAÑANA,MEDELLÍN,OFICIAL,INST EDUC SAN AGUSTIN,INST EDUC SAN AGUSTIN,S


In [30]:
query = """

SELECT *

FROM student_info AS t

INNER JOIN school_info AS s
    ON t.school_campus_id = s.school_campus_id

LIMIT 10

"""

df = conn.execute(query).df()
df

,student_id,estu_agregado,school_institution_id,school_campus_id,estu_dedicacioninternet,estu_dedicacionlecturadiaria,estu_depto_reside,estu_discapacidad,estu_estudiante,estu_etnia,...,cole_cod_mcpio_ubicacion,cole_codigo_icfes,cole_depto_ubicacion,cole_genero,cole_jornada,cole_mcpio_ubicacion,cole_naturaleza,cole_nombre_establecimiento,cole_nombre_sede,cole_sede_principal
0,SB11201520057929,S,170823000106,170823000106,None,None,SUCRE,None,ESTUDIANTE,None,...,70823,15586,SUCRE,MIXTO,MAÑANA,TOLÚ VIEJO,OFICIAL,I.E. HERIBERTO GARCIA GARRIDO,I.E. HERIBERTO GARCIA GARRIDO - SEDE PRINCIPAL,S
1,SB11201520049282,S,168001000550,168001000550,None,None,SANTANDER,None,ESTUDIANTE,None,...,68001,14167,SANTANDER,MIXTO,MAÑANA,BUCARAMANGA,OFICIAL,INSTITUCIÓN EDUCATIVA AURELIO MARTÍNEZ MUTIS,I E ACAD AURELIO MARTINEZ MUTIS,S
2,SB11201520208742,S,125001000231,125001000231,None,None,CUNDINAMARCA,None,ESTUDIANTE,None,...,25001,26922,CUNDINAMARCA,MIXTO,MAÑANA,AGUA DE DIOS,OFICIAL,INSTITUCION EDUCATIVA DEPARTAMENTAL SALESIANO...,INSTITUCION EDUCATIVA DEPARTAMENTAL SALESIANO ...,S
3,SB11201520231149,S,308001009863,308001009863,None,None,ATLANTICO,None,ESTUDIANTE,None,...,8001,718213,ATLANTICO,MIXTO,UNICA,BARRANQUILLA,OFICIAL,INSTITUCION EDUCATIVA DISTRITAL LAS MERCEDES S...,INSTITUCION EDUCATIVA DISTRITAL LAS MERCEDES S...,S
4,SB11201520226424,S,305890001390,305890001390,None,None,ANTIOQUIA,None,ESTUDIANTE,None,...,5890,106146,ANTIOQUIA,MIXTO,COMPLETA,YOLOMBO,NO OFICIAL,INSTITUTO CODESARROLLO,INSTITUTO CODESARROLLO - SEDE PRINCIPAL,S
5,SB11201520068288,S,125572000033,125572000033,None,None,CUNDINAMARCA,None,ESTUDIANTE,None,...,25572,236265,CUNDINAMARCA,MIXTO,UNICA,PUERTO SALGAR,OFICIAL,INSTITUCION EDUCATIVA DEPARTAMENTAL POLICARPA ...,INSTITUCION EDUCATIVA DEPARTAMENTAL POLICARPA ...,S
6,SB11201520143995,S,311848000448,311848000448,None,None,BOGOTA,None,ESTUDIANTE,None,...,11001,22350,BOGOTA,FEMENINO,COMPLETA,BOGOTÁ D.C.,NO OFICIAL,CENT MARIA AUXILIADORA ...,CENT MARIA AUXILIADORA,S
7,SB11201520232839,S,120001791767,120001791767,None,None,CESAR,None,ESTUDIANTE,Cancuamo,...,20001,735613,CESAR,MIXTO,UNICA,VALLEDUPAR,OFICIAL,I.E. RICARDO GONZÁLEZ,I.E. RICARDO GONZÁLEZ - SEDE PRINCIPAL,S
8,SB11201520037063,S,305890001390,305890001390,None,None,ANTIOQUIA,None,ESTUDIANTE,None,...,5890,106146,ANTIOQUIA,MIXTO,COMPLETA,YOLOMBO,NO OFICIAL,INSTITUTO CODESARROLLO,INSTITUTO CODESARROLLO - SEDE PRINCIPAL,S
9,SB11201520576480,S,123001003741,123001003741,None,None,CORDOBA,None,ESTUDIANTE,None,...,23001,96966,CORDOBA,MIXTO,MAÑANA,MONTERÍA,OFICIAL,IE LA PRADERA,IE LA PRADERA,S
